# Week 28, Day 5: Measuring activity II: imaging, fMRI and EEG

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w028/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: From fluorescence back to spikes

Simulate 60 s of a neuron firing Poisson spikes at 2 Hz, imaged at 30 frames per second with an indicator decaying with τ = 0.5 s and noise of sd 0.3 (local rng seed 6). Compute `F`, deconvolve it into `s_hat` (clipping at zero), and compare the correlation of `F` and `s_hat` with the true spikes per frame, `s`, as `corr_raw` and `corr_deconv`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r6 = np.random.default_rng(6)
fps, T, tau = 30, 60, 0.5
nf = fps * T; gamma = np.exp(-1 / (fps * tau))
s = r6.poisson(2.0 / fps, nf).astype(float)        # spikes per frame
noise = 0.3 * r6.normal(size=nf)
c = np.zeros(nf)
for t_ in range(1, nf):
    c[t_] = ...
F = c + noise
s_hat = np.zeros(nf)
s_hat[1:] = ...                                    # F_t - gamma F_{t-1}, clipped at zero
corr_raw = np.corrcoef(F, s)[0, 1]
corr_deconv = np.corrcoef(s_hat, s)[0, 1]
print(f"correlation with true spikes: raw fluorescence {corr_raw:.2f}, deconvolved {corr_deconv:.2f}")
tt = np.arange(nf) / fps
plt.plot(tt[:300], F[:300], label="fluorescence"); plt.plot(tt[:300], s[:300] * 2, label="spikes (x2)"); plt.legend(); plt.xlabel("s"); plt.show()

In [ ]:
import numpy as np
def test_model():
    "F follows the autoregressive calcium model plus noise"
    c = np.zeros(len(s))
    for t_ in range(1, len(s)): c[t_] = gamma * c[t_ - 1] + s[t_]
    assert np.allclose(F - c, noise)

def test_deconv():
    "deconvolution correlates better with the true spikes than raw fluorescence does, and above 0.6"
    assert corr_deconv > 0.6 and corr_deconv > corr_raw

test_model()
test_deconv()
print("All checks passed")

### Exercise: A block design and its GLM

Build a double-gamma HRF on a 0 to 30 s grid, h(t) = t⁵e^(−t)/5! − t¹⁵e^(−t)/(6·15!). For 300 s of 20 s on and 20 s off blocks sampled every 1 s (TR), convolve the boxcar with the HRF to get the regressor, simulate a voxel as 2 × regressor + noise (sd 1, local rng seed 7), fit an intercept and the regressor by least squares (`beta`), compute the t statistic of the effect, and compare R² with a model that uses the raw boxcar instead.

In [ ]:
from math import factorial
th = np.arange(0, 30, 1.0)
hrf = ...                                         # double-gamma haemodynamic response
hrf = hrf / hrf.max()
t_scan = np.arange(300)
box = ((t_scan // 20) % 2 == 1).astype(float)    # 20 s off, 20 s on
reg = np.convolve(box, hrf)[:300]
y = 2.0 * reg + np.random.default_rng(7).normal(size=300)

def fit(x):
    X = np.c_[np.ones(300), x]
    b = np.linalg.lstsq(X, y, rcond=None)[0]
    res = y - X @ b
    s2 = res @ res / (300 - 2)
    se = np.sqrt(s2 * np.linalg.inv(X.T @ X)[1, 1])
    return b, b[1] / se, 1 - res @ res / np.sum((y - y.mean()) ** 2)

beta, t_stat, r2_hrf = fit(reg)
_, _, r2_box = ...
print(f"effect {beta[1]:.2f} (true 2.0), t = {t_stat:.1f};  R2 with HRF {r2_hrf:.3f}, with raw boxcar {r2_box:.3f}")
plt.plot(t_scan, y, lw=0.6); plt.plot(t_scan, beta[0] + beta[1] * reg); plt.xlabel("s"); plt.show()

In [ ]:
import numpy as np
def test_hrf():
    "the HRF peaks between 4 and 6 s and has a negative undershoot"
    assert 4 <= th[np.argmax(hrf)] <= 6 and hrf.min() < 0

def test_beta():
    "the estimated effect is within 0.3 of the true 2.0 and its t statistic above 5"
    assert abs(beta[1] - 2.0) < 0.3 and t_stat > 5

def test_better():
    "the HRF-convolved model explains more variance than the raw boxcar"
    assert r2_hrf > r2_box

test_hrf()
test_beta()
test_better()
print("All checks passed")

### Exercise: Why the EEG needs synchrony

Sinusoids of the same frequency add like phasors (Week 11): N unit sources with phases φ sum to a sinusoid of amplitude |Σ exp(iφ)|. Compute this amplitude for N = 10,000 sources all in phase (`amp_sync`), and the root-mean-square amplitude over 200 sets of random phases (local rng seed 8) (`amp_random`).

In [ ]:
N = 10000
r8 = np.random.default_rng(8)
amp_sync = ...                                    # all phases zero
amps = [abs(np.exp(1j * r8.uniform(0, 2 * np.pi, N)).sum()) for _ in range(200)]
amp_random = ...                                  # root mean square over the 200 draws
print(f"in phase: {amp_sync:.0f};  random phases: {amp_random:.0f} (single draws from {min(amps):.0f} to {max(amps):.0f});  ratio {amp_sync / amp_random:.0f}")

In [ ]:
import numpy as np
def test_amps():
    "the in-phase amplitude is N; the root-mean-square random-phase amplitude is within 10% of sqrt(N)"
    assert abs(amp_sync - N) < 1e-6 * N and abs(amp_random / np.sqrt(N) - 1) < 0.1

test_amps()
print("All checks passed")